# Example Progression 24 — Companion threads + evaluate-while-waiting (SDK, interactive)

Interactive counterpart of [`wf_examples/wf_example_progression_24.py`](../../wf_examples/wf_example_progression_24.py).
It imports the **exact same** `build_assistant_workflow()` and `run_over_websocket()` from that
script (no duplication), creates the workflow on the dev server, drives one run, and cleans up.

A companion thread polls a lab system in the background while the main thread talks to the caller;
an evaluate-while-waiting edge speaks the result the moment it lands, with **no user message**.

> Driven over `client.runs.stream()`, not a handle. That is not a style choice — the REST driver
> does not advance background work. See [`19_companion_threads.ipynb`](../19_companion_threads.ipynb).

> **Async-first.** Uses `AsyncWorkflowClient` with top-level `await` (the setup cell calls
> `nest_asyncio.apply()`). Every method also exists on the synchronous `WorkflowClient`.

In [ ]:
# This notebook lives one level below notebooks/, where _bootstrap.py lives. Walk up from the
# working dir to find it, add that dir to sys.path, then import it so `import interactly` works
# with no install needed. (Works whether the kernel cwd is notebooks/ or wf_example_notebooks/.)
import os, sys
from pathlib import Path
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()

_notebooks_dir = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / '_bootstrap.py').exists():
        _notebooks_dir = _cand
        break
if _notebooks_dir and str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

import _bootstrap  # noqa: F401 - enables `import interactly` (no install needed)

project_root = Path.cwd()
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f'Loaded .env from: {env_path}')
else:
    print(f'Warning: .env file not found at {env_path}')

In [ ]:
# Interactly credentials are read from environment variables.
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell env always wins.
os.environ.setdefault('INTERACTLY_BASE_URL', 'https://api-dev.interactly.ai/workflows')
os.environ.setdefault('INTERACTLY_TEAM_ID', '67458e762b7d3dc15aaea5b5')
os.environ.setdefault('INTERACTLY_USER_ID', '687b1a4f745c8e6806c98d91')

# The bearer token is a secret — never hardcode it in the notebook.
assert os.environ.get('INTERACTLY_API_KEY'), (
    'Set INTERACTLY_API_KEY in your environment before running this notebook.'
)
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

## 1. Build and upload the workflow

In [ ]:
# Import the builder AND the driver straight from the example script rather than duplicating
# them, so this notebook always runs exactly what the script runs. Importing the module only
# defines functions; its main() is guarded by `if __name__ == "__main__"`.
import importlib, sys
from pathlib import Path

_wf_examples = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / "wf_examples" / "wf_example_progression_24.py").exists():
        _wf_examples = _cand / "wf_examples"
        break
assert _wf_examples, "could not locate wf_examples/"
if str(_wf_examples) not in sys.path:
    sys.path.insert(0, str(_wf_examples))

example = importlib.import_module("wf_example_progression_24")

from interactly import AsyncWorkflowClient

client = AsyncWorkflowClient()
config = example.build_assistant_workflow()
workflow = await client.workflows.create_from_config(config, name=config.workflow_config.name)
WF_ID = workflow.id
print("Created", WF_ID, "—", config.workflow_config.name)

## 2. Drive it over the WebSocket

`run_over_websocket()` is the script's own driver. Watch for `waiting_condition_matched` — that is
the conversation advancing without the caller saying anything.

In [ ]:
await example.run_over_websocket(client, WF_ID)

## 3. Cleanup

In [ ]:
await client.workflows.delete(WF_ID)
await client.close()
print("Workflow", WF_ID, "deleted.")